# nb00 — Colab Setup & Data Download
Mounts Google Drive, creates folder structure, downloads NeurIPS 2021 benchmark h5ad files.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

BASE     = Path('/content/drive/MyDrive/multiomics-relationship-modeling')
DATA_DIR = BASE / 'data' / 'benchmark'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Folder ready:', DATA_DIR)

Folder ready: /content/drive/MyDrive/multiomics-relationship-modeling/data/benchmark


In [4]:
import urllib.request
import gzip
import shutil

BASE_URL = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE194nnn/GSE194122/suppl'

FILES = {
    'cite': {
        'gz':   'GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad.gz',
        'h5ad': 'GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad',
        'desc': 'CITE-seq (RNA + protein ADT, ~70k cells)',
    },
    'multiome': {
        'gz':   'GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad.gz',
        'h5ad': 'GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad',
        'desc': 'Multiome (RNA + ATAC, ~69k cells)',
    },
}


def progress(block, block_size, total):
    done = block * block_size
    if total > 0:
        pct = min(100, done * 100 // total)
        bar = '█' * (pct // 5) + '░' * (20 - pct // 5)
        print(f'\r  [{bar}] {pct:3d}%  {done/1e6:.1f}/{total/1e6:.1f} MB',
              end='', flush=True)


def download_and_unzip(key):
    info    = FILES[key]
    gz_path = DATA_DIR / info['gz']
    h5_path = DATA_DIR / info['h5ad']

    print(f"\n── {info['desc']} ──")

    # Download .gz
    if gz_path.exists() or h5_path.exists():
        print(f'  .gz or .h5ad already exists — skipping download.')
    else:
        url = f'{BASE_URL}/{info["gz"]}'
        print(f'  Downloading: {url}')
        urllib.request.urlretrieve(url, gz_path, reporthook=progress)
        print(f'\n  Saved: {gz_path.name}  ({gz_path.stat().st_size/1e9:.2f} GB)')

    # Decompress
    if h5_path.exists():
        print(f'  Already decompressed ({h5_path.stat().st_size/1e9:.2f} GB) — skipping.')
    else:
        print(f'  Decompressing ...')
        with gzip.open(gz_path, 'rb') as f_in, open(h5_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        print(f'  Done: {h5_path.name}  ({h5_path.stat().st_size/1e9:.2f} GB)')
        gz_path.unlink()
        print('  Removed .gz to save Drive space.')


download_and_unzip('cite')
download_and_unzip('multiome')


── CITE-seq (RNA + protein ADT, ~70k cells) ──
  .gz or .h5ad already exists — skipping download.
  Already decompressed (0.62 GB) — skipping.

── Multiome (RNA + ATAC, ~69k cells) ──
  .gz or .h5ad already exists — skipping download.
  Already decompressed (3.12 GB) — skipping.


In [5]:
# Verify
for key, info in FILES.items():
    p = DATA_DIR / info['h5ad']
    status = f'{p.stat().st_size/1e9:.2f} GB' if p.exists() else 'MISSING'
    print(f"{info['desc']:45s}  {status}")

CITE-seq (RNA + protein ADT, ~70k cells)       0.62 GB
Multiome (RNA + ATAC, ~69k cells)              3.12 GB
